ConvNextバージョン

In [3]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
# IMG_ROOT = f'{pre}/data/26_0413_clean'
# IMG_ROOT = f'{pre}/data/26_0413_moreclean'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'
# TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp_clean/train_max_upsampled.csv'
# VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp_clean/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/nomal/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"
# SAVE_DIR = f"../save/clean/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"
# SAVE_DIR = f"../save/moreclean/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"



# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Reduction ratio calculation with a fallback to ensure at least 1 channel
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. 融合モデル (ConvNeXt ベース) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # ConvNeXt をロード (事前学習済みモデルを使用)
        # 今回は軽量な convnext_tiny を使用しますが、必要に応じて base 等に変更してください
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        
        # ConvNeXt の特徴抽出部分のみを使用 (classifier の手前まで)
        self.dl_extractor = convnext.features
        
        # ConvNeXt-Tiny の出力チャンネル数は 768
        dl_out_channels = 768
        
        # 数値データのプロジェクター (出力チャンネルを ConvNeXt の出力に合わせる)
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.ReLU(inplace=True)
        )
        
        # 融合のための Attention モジュール
        self.fusion_attention = CBAM(dl_out_channels)
        
        # 最終分類器
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        # 画像特徴の抽出: Y.shape -> (B, 768, H', W')
        Y = self.dl_extractor(img)
        B, C, H, W = Y.shape
        
        # 数値特徴のプロジェクションと拡張: X.shape -> (B, 768, H', W')
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        # AFF (Attentional Feature Fusion)
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        # 分類
        return self.classifier(fused)

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        # StandardScalerの処理
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            # テスト時は学習データのScalerを使用
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            # 画像読み込みエラー時のダミー
            print(f"Error loading: {img_path}")
            # ConvNeXtに合わせてデフォルトサイズを少し大きく（224）しておくのも良いです
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 5. メイン学習関数 ---
def train_model():
    # ハイパーパラメータ
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    # 画像変換定義 (ConvNeXt 推奨の 224x224 に変更)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    
    # ★ 指定されたCSVを読み込む
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print("-" * 30)
    print(f"Train Data (from {os.path.basename(TRAIN_CSV_PATH)}): {len(train_df)} samples")
    print(f"Val Data   (from {os.path.basename(VAL_CSV_PATH)}):   {len(val_df)} samples")
    print("-" * 30)
    
    # Dataset作成
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    # 学習ループ
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        # 検証ループ
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # ベストモデル保存
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        # グラフ保存
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

# --- 実行 ---
if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
------------------------------
Train Data (from train_max_upsampled.csv): 1200 samples
Val Data   (from val_fixed.csv):   144 samples
------------------------------
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.0194 | Val Acc: 0.5347
  --> Best Model Saved (Acc: 0.5347)
Epoch 2/100 | Loss: 0.8905 | Val Acc: 0.6042
  --> Best Model Saved (Acc: 0.6042)
Epoch 3/100 | Loss: 0.8236 | Val Acc: 0.6111
  --> Best Model Saved (Acc: 0.6111)
Epoch 4/100 | Loss: 0.7967 | Val Acc: 0.6389
  --> Best Model Saved (Acc: 0.6389)
Epoch 5/100 | Loss: 0.7364 | Val Acc: 0.6181
Learning curve saved to ../save/nomal/ConvNext_100epoch_fixed_dataset/learning_curve.png
Epoch 6/100 | Loss: 0.6584 | Val Acc: 0.6319
Epoch 7/100 | Loss: 0.5931 | Val Acc: 0.6389
Epoch 8/100 | Loss: 0.5037 | Val Acc: 0.6042
Epoch 9/100 | Loss: 0.4384 | Val Acc: 0.6250
Epoch 10/100 | Loss: 0.3947 | Val Acc: 0.6528
  --> Best Model Saved (Acc: 0.6528)
Learning curve saved to ../save/nomal

CBAMの初期値変更

In [7]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"

# --- 1. CBAM モジュール (3:7 初期化版) ---
# --- 2. 融合モデル (ConvNeXt ベース) ---
# --- 修正版 CBAM ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.3):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 # より慎重に初期化

    def forward(self, x):
        # 1. Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        # 2. Spatial Attention
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        # チャネルと空間の両方を掛け合わせた重みを返す
        return ch_att * sp_att

# --- 修正版 FusionModel ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        dl_out_channels = 768
        
        # スケールを合わせるために BatchNorm を追加
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels), # 正規化を追加
            nn.ReLU(inplace=True)
        )
        
        # CNN特徴側にも BatchNorm を入れると安定します
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.3)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels), # 分類前にも追加
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y) # 正規化
        
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 融合
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)
# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- (前略: クラス定義やデータセット部分は変更なし) ---

# --- 4. Early Stopping クラス ---
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        """
        Args:
            patience (int): 改善が見られなくなってから何エポック待つか
            verbose (bool): メッセージを表示するかどうか
            delta (float): 改善とみなす最小の変化量
            path (str): ベストモデルの保存先
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta
        self.path = path

    def __call__(self, val_acc, model):
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        '''バリデーション精度が向上したときにモデルを保存'''
        if self.verbose:
            print(f'Validation accuracy increased ({self.val_acc_max:.4f} --> {val_acc:.4f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

# --- (前略: クラス定義やデータセット部分は変更なし) ---

# --- 4. Early Stopping クラス ---
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        """
        Args:
            patience (int): 改善が見られなくなってから何エポック待つか
            verbose (bool): メッセージを表示するかどうか
            delta (float): 改善とみなす最小の変化量
            path (str): ベストモデルの保存先
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta
        self.path = path

    def __call__(self, val_acc, model):
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        '''バリデーション精度が向上したときにモデルを保存'''
        if self.verbose:
            print(f'Validation accuracy increased ({self.val_acc_max:.4f} --> {val_acc:.4f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

# --- 5. メイン学習関数 (Early Stopping 導入版) ---
def train_model():
    # ハイパーパラメータ
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    PATIENCE = 20  # 10エポック改善しなければ終了
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    # 画像変換・データロード (中略)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    # Early Stopping の初期化
    early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path=best_model_path)
    
    history = {'train_loss': [], 'val_acc': []}

    print(f"Starting Training with Early Stopping (patience={PATIENCE}) on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)

        # --- Validation Phase ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Early Stopping の判定
        early_stopping(val_acc, model)
        
        if early_stopping.early_stop:
            print("Early stopping triggered. Training finished.")
            break
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {early_stopping.val_acc_max:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Training with Early Stopping (patience=20) on cuda...
Epoch 1/100 | Loss: 1.0382 | Val Acc: 0.4583
Validation accuracy increased (0.0000 --> 0.4583).  Saving model ...
Epoch 2/100 | Loss: 0.8316 | Val Acc: 0.5139
Validation accuracy increased (0.4583 --> 0.5139).  Saving model ...
Epoch 3/100 | Loss: 0.7145 | Val Acc: 0.5417
Validation accuracy increased (0.5139 --> 0.5417).  Saving model ...
Epoch 4/100 | Loss: 0.5968 | Val Acc: 0.5278
EarlyStopping counter: 1 out of 20
Epoch 5/100 | Loss: 0.4550 | Val Acc: 0.5278
EarlyStopping counter: 2 out of 20
Epoch 6/100 | Loss: 0.3271 | Val Acc: 0.4931
EarlyStopping counter: 3 out of 20
Epoch 7/100 | Loss: 0.3429 | Val Acc: 0.5139
EarlyStopping counter: 4 out of 20
Epoch 8/100 | Loss: 0.2389 | Val Acc: 0.5556
Validation accuracy increased (0.5417 --> 0.5556).  Saving model ...
Epoch 9/100 | Loss: 0.2125 | Val Acc: 0.5000
EarlyStopping counter: 1 out of 20
Epoch 10/100 | Loss: 0.2065 | Val Acc: 0.4514
EarlyStopping counter: 2 out of 20


過学習対策の方から過学習対策抜いたの，最新のプログラムと同義

In [1]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/nomal/ConvNext_{TARGET_EPOCH}epoch_baseline"

# --- 1. CBAM モジュール (初期値 5:5) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        return ch_att * sp_att

# --- 2. 融合モデル (Dropoutなし) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        dl_out_channels = 768
        
        # Dropoutを削除
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        
        # Dropoutを削除
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 5. メイン学習関数 (対策なしベースライン) ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    # データ拡張なし (訓練・検証ともに同じ)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    
    # AdamW ではなく、通常の Adam (Weight Decay なし)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    # スケジューラなし
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Baseline Training (No Overfitting Countermeasures) on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # ベストモデルの保存
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Baseline Training (No Overfitting Countermeasures) on cuda...
Epoch 1/100 | Train Loss: 0.9898 | Val Loss: 1.1110 | Val Acc: 0.4861
  --> Best Model Saved! (Acc: 0.4861)
Epoch 2/100 | Train Loss: 0.8296 | Val Loss: 1.0794 | Val Acc: 0.5278
  --> Best Model Saved! (Acc: 0.5278)
Epoch 3/100 | Train Loss: 0.7230 | Val Loss: 1.0451 | Val Acc: 0.5208
Epoch 4/100 | Train Loss: 0.6071 | Val Loss: 1.1345 | Val Acc: 0.5069
Epoch 5/100 | Train Loss: 0.4229 | Val Loss: 1.2536 | Val Acc: 0.4931
Epoch 6/100 | Train Loss: 0.3325 | Val Loss: 1.7868 | Val Acc: 0.4306
Epoch 7/100 | Train Loss: 0.2856 | Val Loss: 1.5716 | Val Acc: 0.4792
Epoch 8/100 | Train Loss: 0.2408 | Val Loss: 1.6597 | Val Acc: 0.5000
Epoch 9/100 | Train Loss: 0.1947 | Val Loss: 1.7102 | Val Acc: 0.4861
Epoch 10/100 | Train Loss: 0.1956 | Val Loss: 1.7860 | Val Acc: 0.4653
Epoch 11/100 | Train Loss: 0.1301 | Val Loss: 1.7923 | Val Acc: 0.5000
Epoch 12/100 | Train Loss: 0.1703 | Val Loss: 1.6298 | Val Acc: 0.4931
Epoch 13/1